In [1]:
# Import libraries
import numpy as np
import pandas as pd
import requests
import pdfkit
import jinja2
import base64
import os

# ============= CONFIGURATION VARIABLES =============
ACADEMIC_SESSION = '2025-2026'
EXAM_NAME = 'Term I'

# Define optional subject groups (students take one from each group)
OPTIONAL_SUBJECT_GROUPS = [
    ['Hindi', 'Assamese'],  # Student takes either Hindi OR Assamese
    # Add more optional groups here if needed, e.g.:
    # ['Sanskrit', 'Arabic'],
]
# ==================================================

# Import data from excel files
df1 = pd.read_excel('Formative Assessment I_class_VIII.xlsx')
# Update all values of Exam Name to FA-I
df1['Exam Name'] = 'FA-I'

df2 = pd.read_excel('Formative Assessment-II_class_VIII.xlsx')
# Update all values of Exam Name to FA-II
df2['Exam Name'] = 'FA-II'

df3 = pd.read_excel('Half Yearly Exam_class_VIII.xlsx')
# Update all values of Exam Name to Half Yearly
df3['Exam Name'] = 'Half Yearly'

# In df3, where the Full Mark is not 100, multiply the Full Mark, Pass Mark and Marks Secured by 100/Full Mark
df3.loc[df3['Full Mark']!=100, 'Pass Mark'] = df3['Pass Mark']*100/df3['Full Mark']
df3.loc[df3['Full Mark']!=100, 'Marks Secured'] = df3['Marks Secured']*100/df3['Full Mark']
df3.loc[df3['Full Mark']!=100, 'Full Mark'] = df3['Full Mark']*100/df3['Full Mark']

# Merge the dataframes
df = pd.concat([df1, df2, df3], ignore_index=True)

# Print diagnostics
print('Number of cells with value 0 in marks secured column:', df[df['Marks Secured']==0].shape[0])
print('Student names with value 0 in marks secured column:', df[df['Marks Secured']==0]['Student Name'].unique())

# Get all the Subject names
subjects = df['Subject Name'].unique()

# Get all the students
students = df['Student Name'].unique()

# Get all the Exam names
exams = df['Exam Name'].unique()

# Print summary
print('Number of Subjects:', len(subjects))
print('Subjects:', subjects)
print('Number of Students:', len(students))
print('Students:', students)
print('Number of Exams:', len(exams))
print('Exams:', exams)

# Create a new dataframe for students data
df_new = pd.DataFrame(students, columns=['Student Name'])

# Loop through all the students
for student in students:
    # Find the first row of the student in the dataframe
    row = df.loc[df['Student Name'] == student].iloc[0]
    # Add the Class, Section, Roll No to the new dataframe
    df_new.loc[df_new['Student Name'] == student, 'Class'] = row['Class']
    df_new.loc[df_new['Student Name'] == student, 'Section'] = row['Section']
    df_new.loc[df_new['Student Name'] == student, 'Roll No'] = row['Roll No']
    df_new.loc[df_new['Student Name'] == student, 'Profile Photo URL'] = row['profile_photo_url']

# Add columns for each exam and subject
for exam in exams:
    for subject in subjects:
        df_new[exam + '_' + subject] = 0
        df_new[exam + '_' + subject + '_total'] = 0
        df_new[exam + '_' + subject + '_pass'] = 0

# Add marks to the new dataframe
for student in students:
    for exam in exams:
        for subject in subjects:
            # check if the subject is present in the exam
            if df.loc[(df['Exam Name'] == exam) & (df['Subject Name'] == subject)].empty:
                continue
            
            # full marks
            full_mark = df.loc[(df['Exam Name'] == exam) & (df['Subject Name'] == subject), 'Full Mark'].iloc[0]
            
            # pass marks
            pass_mark = df.loc[(df['Exam Name'] == exam) & (df['Subject Name'] == subject), 'Pass Mark'].iloc[0]
            
            # Check if the row of the student exists in the dataframe
            if df.loc[(df['Student Name'] == student) & (df['Exam Name'] == exam) & (df['Subject Name'] == subject)].empty:
                # If it doesn't exist, add 0 to the new dataframe
                df_new.loc[df_new['Student Name'] == student, exam + '_' + subject] = 0
                df_new.loc[df_new['Student Name'] == student, exam + '_' + subject + '_total'] = full_mark
                df_new.loc[df_new['Student Name'] == student, exam + '_' + subject + '_pass'] = pass_mark
            else:
                # If it exists, get the row
                row = df.loc[(df['Student Name'] == student) & (df['Exam Name'] == exam) & (df['Subject Name'] == subject)].iloc[0]
                # Add the marks to the new dataframe
                df_new.loc[df_new['Student Name'] == student, exam + '_' + subject] = row['Marks Secured']
                df_new.loc[df_new['Student Name'] == student, exam + '_' + subject + '_total'] = full_mark
                df_new.loc[df_new['Student Name'] == student, exam + '_' + subject + '_pass'] = pass_mark

# Add weightage column for each subject
for subject in subjects:
    df_new['Weightage ' + subject] = np.nan

# Calculate the weightage for each subject by taking the 40% from FA-I, 40% from FA-II and 80% from Half Yearly
for subject in subjects:
    # If the subject is not present in FA1, FA2
    if df_new['FA-I_' + subject].sum() == 0 or df_new['FA-II_' + subject].sum() == 0:
        # Calculate the weightage for each subject by taking the 100% from Half Yearly
        df_new['Weightage ' + subject] = round(df_new['Half Yearly_' + subject] * 1, 2)
        # Calculate the full weightage for each subject by taking the 100% from Half Yearly
        df_new['Weightage ' + subject + ' total'] = df_new['Half Yearly_' + subject + '_total'] * 1
    else:
        df_new['Weightage ' + subject] = round(df_new['FA-I_' + subject] * 0.4 + df_new['FA-II_' + subject] * 0.4 + df_new['Half Yearly_' + subject] * 0.8, 2)
        # Calculate the full weightage for each subject by taking the 40% from FA1, 40% from FA2 and 80% from Half Yearly
        df_new['Weightage ' + subject + ' total'] = df_new['FA-I_' + subject + '_total'] * 0.4 + df_new['FA-II_' + subject + '_total'] * 0.4 + df_new['Half Yearly_' + subject + '_total'] * 0.8

# Calculate the weightage result for each subject, pass if weightage is greater than 30% of the full weightage
for subject in subjects:
    # calculate the weightage result
    df_new['Result ' + subject] = np.where(df_new['Weightage ' + subject] >= df_new['Weightage ' + subject + ' total'] * 0.3, 'Pass', 'Fail')

# ============= HELPER FUNCTIONS FOR OPTIONAL SUBJECTS =============

def has_attempted_subject(row, subject):
    """Check if a student has attempted a subject in any exam"""
    return (row.get('FA-I_' + subject, 0) > 0 or 
            row.get('FA-II_' + subject, 0) > 0 or 
            row.get('Half Yearly_' + subject, 0) > 0)

def should_count_subject(row, subject):
    """Determine if a subject should be counted for this student"""
    # Check if subject is in any optional group
    for group in OPTIONAL_SUBJECT_GROUPS:
        if subject in group:
            # It's an optional subject - only count if student attempted it
            return has_attempted_subject(row, subject)
    
    # Not an optional subject - always count
    return True

def calculate_student_total_weightage(row):
    """Calculate total weightage for subjects the student actually took"""
    total = 0
    for subject in subjects:
        if should_count_subject(row, subject):
            total += row['Weightage ' + subject]
    return total

def calculate_student_total_marks(row):
    """Calculate total possible marks for subjects the student actually took"""
    total = 0
    for subject in subjects:
        if should_count_subject(row, subject):
            total += row['Weightage ' + subject + ' total']
    return total

# ============= APPLY CALCULATIONS =============

# Calculate totals using the helper functions
df_new['Total Weightage'] = df_new.apply(calculate_student_total_weightage, axis=1)
df_new['Total Marks'] = df_new.apply(calculate_student_total_marks, axis=1)

# Calculate the percentage
df_new['Percentage'] = df_new['Total Weightage'] / df_new['Total Marks'] * 100

# Add a result column
df_new['Result'] = ''

# Pass if total weightage is greater than 30% of total full marks
df_new.loc[df_new['Total Weightage'] >= df_new['Total Marks'] * 0.3, 'Result'] = 'Pass'

# Fail if total weightage is less than 30% of total full marks
df_new.loc[df_new['Total Weightage'] < df_new['Total Marks'] * 0.3, 'Result'] = 'Fail'

# Calculate the rank
df_new['Rank'] = df_new['Percentage'].rank(ascending=False)

# Export the dataframe to excel
df_new.to_excel('Result.xlsx', index=False)

print("\n✓ Results calculated successfully!")
print(f"✓ Optional subject groups configured: {OPTIONAL_SUBJECT_GROUPS}")
print("✓ Students taking optional subjects will only be evaluated on subjects they attempted")

# ============= PDF GENERATION =============

# Create results directory if it doesn't exist
if not os.path.exists('results'):
    os.makedirs('results')

# PDF Options
options = {
    'page-size': 'A4',
    'margin-top': '0.25in',
    'margin-right': '0.35in',
    'margin-bottom': '0.25in',
    'margin-left': '0.35in',
}

# Images to base64
images = {
    'seba_logo': base64.b64encode(open('./../seba_logo.png', 'rb').read()).decode(),
    'school_logo': base64.b64encode(open('./../school_logo.png', 'rb').read()).decode(),
    'sig_examiner': base64.b64encode(open('./../sig_examiner.png', 'rb').read()).decode(),
    'sig_principal': base64.b64encode(open('./../sig_principal.png', 'rb').read()).decode(),
    'default_profile': base64.b64encode(open('./../profile.png', 'rb').read()).decode(),
}

# Exams dictionary
exams_dict = {
    'FA-I': 'Formative Assessment - I (40%)',
    'FA-II': 'Formative Assessment - II (40%)',
    'Half Yearly': 'Half Yearly Exam (80%)',
}

# Helper function to get subjects a student actually took
def get_student_subjects(student_row, all_subjects):
    """Get list of subjects the student actually took"""
    student_subjects = []
    for subject in all_subjects:
        if should_count_subject(student_row, subject):
            student_subjects.append(subject)
    return student_subjects

# Sort students by rank before generating PDFs
df_new_sorted = df_new.sort_values('Rank')
students_sorted = df_new_sorted['Student Name'].tolist()

# Generate PDFs for each student (in rank order)
for student in students_sorted:
    # Get the student data
    student_data = df_new.loc[df_new['Student Name'] == student].iloc[0]
    
    # Get the student data as dictionary
    student_data_dict = student_data.to_dict()

    # Get the profile photo URL
    profile_photo_url = student_data_dict['Profile Photo URL']

    profile_photo_base64 = ''
    # Check if URL exists and is not empty
    if pd.notna(profile_photo_url) and str(profile_photo_url).strip() != '':
        try:
            # Prefix the URL with the base path
            full_photo_url = 'https://avant-api.epmsbirkuchi.com/storage/' + str(profile_photo_url)
            
            # Download the image
            response = requests.get(full_photo_url, timeout=10)
            if response.status_code == 200:
                # Convert to base64
                profile_photo_base64 = 'data:image/jpeg;base64,' + base64.b64encode(response.content).decode()
            else:
                print(f"Failed to download photo for {student}: HTTP {response.status_code}, using default")
                # Use default profile picture
                profile_photo_base64 = 'data:image/png;base64,' + images['default_profile']
        except Exception as e:
            print(f"Error loading profile photo for {student}: {e}, using default")
            # Use default profile picture
            profile_photo_base64 = 'data:image/png;base64,' + images['default_profile']
    else:
        # No URL provided, use default profile picture
        profile_photo_base64 = 'data:image/png;base64,' + images['default_profile']

    context = {
        'subjects': get_student_subjects(student_data, subjects),
        'exams': exams_dict,
        'data': student_data_dict,
        'academic_session': ACADEMIC_SESSION,
        'exam_name': EXAM_NAME,
        'profile_photo': profile_photo_base64,
        'seba_logo': 'data:image/png;base64,' + images['seba_logo'],
        'school_logo': 'data:image/png;base64,' + images['school_logo'],
        'sig_examiner': 'data:image/png;base64,' + images['sig_examiner'],
        'sig_principal': 'data:image/png;base64,' + images['sig_principal'],
    }

    # Create template
    templateLoader = jinja2.FileSystemLoader(searchpath="./../")
    templateEnv = jinja2.Environment(loader=templateLoader)
    template = templateEnv.get_template("result_template.html")
    
    # Render the template
    output_text = template.render(context)

    output_file_name = 'results/Rank-' + str(int(student_data_dict['Rank'])) + '_Roll-' + str(int(student_data_dict['Roll No'])) + '_' + student + '.pdf'
    
    # Create pdf
    try:
        pdfkit.from_string(output_text, output_file_name, options=options)
        print(f"✓ Generated PDF for {student}")
    except Exception as e:
        print(f"✗ Error generating PDF for {student}: {e}")

print("\n✓ All PDFs generated successfully!")

Number of cells with value 0 in marks secured column: 99
Student names with value 0 in marks secured column: ['Shruti Das' 'Jeson  Rongphi' 'Kunal boro' 'Kripamoni Tumung'
 'Koyel Biswas' 'Subham Chetri' 'Abhijit Das' 'Rajkumar Das'
 'Nishanta Basumatary' 'Priya kumari Singh' 'Dhritojit Boro'
 'KASTURI SARMA' 'KAUTAV DEKA' 'Swastika Chanda' 'Mimi Bordoloi'
 'PRAYAS THAPA']
Number of Subjects: 9
Subjects: ['English I' 'English II' 'Hindi' 'Maths' 'Science' 'Social Science'
 'Computer' 'Essay' 'Assamese']
Number of Students: 31
Students: ['Shivraj Kumar' 'Swastika Chanda' 'Shruti Das' 'Mrinmoy Nath'
 'Jeson  Rongphi' 'Kunal boro' 'Kripamoni Tumung' 'Nayan Jyoti Mahanta'
 'Pratik Thapa' 'Subham Chetri' 'Koyel Biswas' 'Lalit kr Nayak'
 'Ranjita Hajong' 'Mimi Bordoloi' 'Abhijit Das' 'Rajkumar Das'
 'Nishanta Basumatary' 'Shivani Ray' 'Tilak Teron' 'Nipshikha Kashyap'
 'Pranavi Sharma' 'Sija Bordoloi' 'Khusi Chetry' 'Mangita Nayak'
 'Priya kumari Singh' 'Sambhav Upadhya' 'Dhritojit Boro' 'DE

/tmp/ipykernel_23792/3720635569.py:75: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'profile_pictures/onetime/120_1757385621.jpeg' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_new.loc[df_new['Student Name'] == student, 'Profile Photo URL'] = row['profile_photo_url']
/tmp/ipykernel_23792/3720635569.py:108: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '7.05' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df_new.loc[df_new['Student Name'] == student, exam + '_' + subject] = row['Marks Secured']
/tmp/ipykernel_23792/3720635569.py:108: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '13.5' has dtype incompatible with int64, please explicitly cast to a compatib


✓ Results calculated successfully!
✓ Optional subject groups configured: [['Hindi', 'Assamese']]
✓ Students taking optional subjects will only be evaluated on subjects they attempted
✓ Generated PDF for KASTURI SARMA
✓ Generated PDF for Ranjita Hajong
✓ Generated PDF for Subham Chetri
✓ Generated PDF for DELFI JAANT
✓ Generated PDF for Swastika Chanda
✓ Generated PDF for Nipshikha Kashyap
✓ Generated PDF for Shivraj Kumar
✓ Generated PDF for Mangita Nayak
✓ Generated PDF for Pranavi Sharma
✓ Generated PDF for Tilak Teron
✓ Generated PDF for KAUTAV DEKA
✓ Generated PDF for Pratik Thapa
✓ Generated PDF for Nayan Jyoti Mahanta
✓ Generated PDF for Kunal boro
✓ Generated PDF for Shivani Ray
✓ Generated PDF for Shruti Das
✓ Generated PDF for Mrinmoy Nath
✓ Generated PDF for Rajkumar Das
✓ Generated PDF for Sambhav Upadhya
✓ Generated PDF for Nishanta Basumatary
✓ Generated PDF for Sija Bordoloi
✓ Generated PDF for Lalit kr Nayak
✓ Generated PDF for Mimi Bordoloi
✓ Generated PDF for PRAYAS T

In [2]:
# Combine all pdfs into one
import os
from os import listdir
from os.path import isfile, join
from PyPDF2 import PdfMerger

# Get all the pdf files
pdf_files = [f for f in listdir('results') if isfile(join('results', f))]

# Remove the result pdf if it exists
if '00_Result_Class_VIII.pdf' in pdf_files:
    pdf_files.remove('00_Result_Class_VIII.pdf')

# Sort the pdf files by rank number (extract rank from filename)
def get_rank(filename):
    """Extract rank number from filename like 'Rank-1_Roll-15_Alice.pdf'"""
    try:
        # Split by 'Rank-' and get the part after it
        rank_part = filename.split('Rank-')[1]
        # Get the number before the first underscore
        rank = int(rank_part.split('_')[0])
        return rank
    except:
        # If parsing fails, return a large number to put it at the end
        return 999999

pdf_files.sort(key=get_rank)

# Create a pdf merger
merger = PdfMerger()

# Add all the pdf files to the merger
for pdf in pdf_files:
    merger.append('results/' + pdf)

# Write the merger to a file
merger.write('results/00_Result_Class_VIII.pdf')

# Close the merger
merger.close()

print(f"✓ Combined {len(pdf_files)} PDFs in rank order")

✓ Combined 31 PDFs in rank order
